In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Butterfly Effect\n",
    "## Sensitivity to Initial Conditions in the Feistel Block Cipher\n",
    "\n",
    "A single bit flip in the plaintext produces a completely different ciphertext.\n",
    "The chaotic round function amplifies tiny changes across the entire block.\n",
    "Tampering with the ciphertext causes decryption to fail integrity checks."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "from network import ChaoticOscillatoryNetwork\n",
    "from block_cipher import CAMCBlockCipher"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "plt.rcParams.update({\n",
    "    'figure.facecolor': '#fafafa',\n",
    "    'axes.facecolor': '#fafafa',\n",
    "    'axes.edgecolor': '#222222',\n",
    "    'axes.labelcolor': '#222222',\n",
    "    'text.color': '#222222',\n",
    "    'font.size': 11,\n",
    "    'axes.grid': True,\n",
    "    'grid.alpha': 0.3,\n",
    "    'grid.color': '#999999'\n",
    "})"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "N = 128\n",
    "SCALE = 1.5\n",
    "ROUNDS = 8\n",
    "STEPS = 2\n",
    "\n",
    "np.random.seed(7)\n",
    "net = ChaoticOscillatoryNetwork(n_neurons=N, weight_scale=SCALE)\n",
    "cipher = CAMCBlockCipher(net, rounds=ROUNDS, steps_per_round=STEPS)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "plain_a = b\"A\" * 64\n",
    "plain_b = bytearray(plain_a)\n",
    "plain_b[0] ^= 0x01\n",
    "plain_b = bytes(plain_b)\n",
    "\n",
    "enc_a = cipher.encrypt(plain_a)\n",
    "enc_b = cipher.encrypt(plain_b)\n",
    "\n",
    "diff_bits = sum(bin(x ^ y).count('1') for x, y in zip(enc_a, enc_b))\n",
    "total_bits = len(enc_a) * 8\n",
    "print(f\"Changed bits: {diff_bits}/{total_bits} = {100*diff_bits/total_bits:.1f}%\")\n",
    "print(f\"Cipher A hex: {enc_a.hex()}\")\n",
    "print(f\"Cipher B hex: {enc_b.hex()}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "dec_a = cipher.decrypt(enc_a)\n",
    "dec_b = cipher.decrypt(enc_b)\n",
    "print(f\"Decrypted A first bytes: {dec_a[:8]}\")\n",
    "print(f\"Decrypted B first bytes: {dec_b[:8]}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Integrity Mode: Tamper Detection"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "pattern = b\"CHKSUM1234CHECKSABCDEFGHIJKLMNOP\"\n",
    "cipher.set_integrity_pattern(pattern)\n",
    "msg = b\"B\" * 32\n",
    "enc = cipher.encrypt(msg)\n",
    "print(f\"Original plaintext: {msg}\")\n",
    "print(f\"Ciphertext (hex): {enc.hex()}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "dec = cipher.decrypt(enc)\n",
    "print(f\"Decrypted: {dec}\")\n",
    "print(f\"Match: {dec == msg}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "tampered = bytearray(enc)\n",
    "tampered[10] ^= 0xFF\n",
    "try:\n",
    "    cipher.decrypt(bytes(tampered))\n",
    "    print(\"ERROR: Tampering undetected\")\n",
    "except ValueError as e:\n",
    "    print(f\"Tampering detected: {e}\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}